In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')


In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.sport-highlights",
        description="Identify brand product placement in the video footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          Analyze the provided video footage to determine if it depicts a soccer highlight, specifically a goal being scored.

          Read the following definitions carefully before making your decision:
          highlight: This scene clearly shows a goal being scored. Visual indicators include the ball crossing the goal line, the ball hitting the back of the net and causing it to bulge/ripple, or a player striking the ball directly past the goalkeeper into the goal. When using this label, the player should be actively hitting the ball and causing it to score.
          other: This applies to any scene that does not clearly show a goal entering the net. This includes standard passing in the midfield, throw-ins, the ball going out of bounds, fouls, general crowd shots, players running without a clear scoring attempt, or any non-soccer footage.

          Task:
          Identify which category best describes the footage. Output ONLY one of the following exact labels: highlight or other. Do not include any other text or explanation.

          CRITICAL: You must not output anything other than exactly "highlight" or "other". Do not include punctuation, markdown formatting, conversational filler, or explanations of any kind.

        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=10,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.sport-highlights:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")